<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Mini_projet_W5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification d'images (Chats vs Chiens) avec Augmentation de Données

In [ ]:
# 1. Chargement des données et générateurs
import os, math, re, random
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

np.random.seed(42); tf.random.set_seed(42)

# Configuration
DATA_ROOT = Path("data/cats_dogs")
train_dir = (DATA_ROOT / "train" / "train") if (DATA_ROOT / "train" / "train").exists() else (DATA_ROOT / "train")
test_dir  = (DATA_ROOT / "test"  / "test")  if (DATA_ROOT / "test"  / "test").exists()  else (DATA_ROOT / "test")

IMG_HEIGHT, IMG_WIDTH = 180, 180
batch_size = 32
seed = 1337

# Fonction utilitaire de construction de DataFrame
def build_df_from_folder(folder: Path, labeled: bool=True):
    exts = ('*.jpg','*.jpeg','*.png','*.bmp')
    files = []
    for ex in exts:
        files.extend(glob(str(folder / '**' / ex), recursive=True))
    if not files:
        print(f"Warning: No images found under {folder}")
        return pd.DataFrame(columns=(["filepath", "label"] if labeled else ["filepath"]))
    rows = []
    for f in files:
        if labeled:
            name = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {"cat","cats"}: label = "cat"
            elif parent in {"dog","dogs"}: label = "dog"
            else:
                if re.search(r'(^|[^a-z])cat([^a-z]|$)', name): label = "cat"
                elif re.search(r'(^|[^a-z])dog([^a-z]|$)', name): label = "dog"
                else: continue
            rows.append({"filepath": f, "label": label})
        else:
            rows.append({"filepath": f})
    return pd.DataFrame(rows)

try:
    df_train_full = build_df_from_folder(train_dir, labeled=True)
    df_test_full  = build_df_from_folder(test_dir,  labeled=False)

    df_tr, df_val = train_test_split(
        df_train_full, test_size=0.2, stratify=df_train_full["label"], random_state=seed
    )

    train_gen = ImageDataGenerator(rescale=1./255, rotation_range=45, width_shift_range=0.15, height_shift_range=0.15, zoom_range=0.5, horizontal_flip=True)
    val_gen = ImageDataGenerator(rescale=1./255)
    test_gen = ImageDataGenerator(rescale=1./255)

    train_flow = train_gen.flow_from_dataframe(df_tr, x_col="filepath", y_col="label", target_size=(IMG_HEIGHT, IMG_WIDTH), class_mode="binary", batch_size=batch_size, shuffle=True, seed=seed, validate_filenames=False)
    val_flow = val_gen.flow_from_dataframe(df_val, x_col="filepath", y_col="label", target_size=(IMG_HEIGHT, IMG_WIDTH), class_mode="binary", batch_size=batch_size, shuffle=False, validate_filenames=False)
    test_flow = test_gen.flow_from_dataframe(df_test_full, x_col="filepath", y_col=None, target_size=(IMG_HEIGHT, IMG_WIDTH), class_mode=None, batch_size=batch_size, shuffle=False, validate_filenames=False)
except Exception as e:
    print(f"Erreur de chargement : {e}. Assurez-vous que le dossier data/cats_dogs est bien structuré.")

## 2. Examiner les données

Nous analysons ici l'équilibre des classes. Si les classes sont équilibrées, le modèle n'aura pas de biais naturel vers l'une ou l'autre. Les variations visuelles (pose, arrière-plan) sont gérées par l'augmentation de données.

In [ ]:
# Analyse de l'équilibre
print(f"Répartition des classes :\n{df_tr['label'].value_counts()}")

# Visualisation d'échantillons
plt.figure(figsize=(10, 10))
for i in range(9):
    img, label = train_flow.next()
    plt.subplot(3, 3, i + 1)
    plt.imshow(img[0])
    plt.title("Chat" if label[0] == 0 else "Chien")
    plt.axis("off")
plt.show()

## 3. Architecture du Modèle CNN

Nous utilisons un modèle séquentiel composé de :
- **Blocs Convolutionnels** : Pour extraire les caractéristiques spatiales.
- **MaxPooling** : Pour réduire la dimensionnalité.
- **Dropout** : Pour prévenir le surapprentissage.
- **Couche de sortie** : Une unité avec activation Sigmoïde pour la classification binaire.

In [ ]:
from tensorflow.keras import layers, models

def build_model():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = build_model()
model.summary()

## 4. Entraînement avec Early Stopping

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    train_flow,
    epochs=20,
    validation_data=val_flow,
    callbacks=[early_stop]
)

## 5. Analyse des performances

In [ ]:
# Courbes d'apprentissage
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Précision')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Perte')
plt.legend()
plt.show()

## 6. Évaluation et Inférence

In [ ]:
# Prédictions sur la validation
y_val_pred = (model.predict(val_flow) > 0.5).astype("int32")
y_val_true = val_flow.classes

print(classification_report(y_val_true, y_val_pred, target_names=val_flow.class_indices.keys()))

# Inférence sur le test set
test_probs = model.predict(test_flow)
test_preds = (test_probs > 0.5).astype("int32")

submission = pd.DataFrame({
    "filepath": df_test_full["filepath"],
    "prob_dog": test_probs.flatten(),
    "pred_label": ["dog" if p == 1 else "cat" for p in test_preds.flatten()]
})

submission.to_csv("predictions_test.csv", index=False)
display(submission.head())